# bronze_ingest  (pipeline task 1)
Ingests StackOverflow `users` for 2021-H1 with **dlt** (data load tool, dlthub):
reads `users.parquet` from the public source URL in streaming batches, keeps users created
between 2021-01-01 and 2021-06-30 and writes Parquet to MinIO:
`s3://bronze/users/2021/users_2021.parquet`.

`write_disposition="replace"` makes every run overwrite the previous file (no accumulation).

In [1]:
import os
import sys

sys.path.insert(0, "/opt/airflow/jobs")

# Keep the raw column names (CreationDate, DisplayName, ...): Bronze is raw data.
os.environ["SCHEMA__NAMING"] = "direct"

import dlt
from dlt.sources.credentials import AwsCredentials

from common import (
    AWS_KEY,
    AWS_REGION,
    AWS_SECRET,
    BRONZE_BUCKET,
    BRONZE_USERS,
    MINIO_ENDPOINT,
    USERS_SOURCE_URL,
    ensure_buckets,
    iter_window_batches,
    list_keys,
    s3_client,
)

YEAR = 2021
FINAL_KEY = BRONZE_USERS[YEAR].split("/", 1)[1]  # users/2021/users_2021.parquet

## dlt resource: streamed + filtered source

In [2]:
@dlt.resource(name="users_2021", write_disposition="replace")
def users_2021():
    print(f"Reading {USERS_SOURCE_URL} (window {YEAR}-01-01 .. {YEAR}-06-30)")
    for table in iter_window_batches("users.parquet", YEAR):
        yield table

## dlt pipeline -> MinIO (filesystem destination, parquet)
The layout makes dlt write `<dataset_name>/2021/users_2021.parquet` = `users/2021/users_2021.parquet`.

In [3]:
ensure_buckets()

destination = dlt.destinations.filesystem(
    bucket_url=f"s3://{BRONZE_BUCKET}",
    credentials=AwsCredentials(
        aws_access_key_id=AWS_KEY,
        aws_secret_access_key=AWS_SECRET,
        endpoint_url=MINIO_ENDPOINT,
        region_name=AWS_REGION,
    ),
    layout="2021/{table_name}.{ext}",
    warn_unsafe_layout_separators=False,  # single table, no prefix collisions possible
)
pipeline = dlt.pipeline(
    pipeline_name="bronze_users_2021",
    destination=destination,
    dataset_name="users",
    pipelines_dir="/tmp/dlt_pipelines",
)
load_info = pipeline.run(users_2021(), loader_file_format="parquet")
print(load_info)

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:859 FutureWarning: section/key [core/sql_alchemy_conn] has been deprecated, you should use[database/sql_alchemy_conn] instead. Please update your `conf.get*` call to use the new name

Reading https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/users.parquet (window 2021-01-01 .. 2021-06-30)


Pipeline bronze_users_2021 load step finished in 0.49 seconds
1 load package(s) were loaded to destination filesystem and into dataset users
The filesystem destination used s3://bronze location to store data
Load package 1790872313.862173 is LOADED and contains no failed jobs


## Clean dlt bookkeeping and verify: exactly one file, at the expected path
dlt writes its own state tables (`_dlt_loads`, `_dlt_version`, `_dlt_pipeline_state`, `init`)
next to the data. They are not needed for `replace` loads, so they are removed to keep
Bronze limited to the data files.

In [4]:
client = s3_client()
for key in list_keys(client, BRONZE_BUCKET, "users/"):
    if key.split("/")[1].startswith("_dlt_") or key == "users/init":
        client.delete_object(Bucket=BRONZE_BUCKET, Key=key)
keys = list_keys(client, BRONZE_BUCKET, "users/2021/")
print("Objects under bronze/users/2021/:", keys)
if keys != [FINAL_KEY]:
    raise RuntimeError(f"Expected exactly [{FINAL_KEY}] but found {keys}")
size_mb = client.head_object(Bucket=BRONZE_BUCKET, Key=FINAL_KEY)["ContentLength"] / 1e6
print(f"s3://{BRONZE_USERS[YEAR]} written ({size_mb:.1f} MB)")

Objects under bronze/users/2021/: ['users/2021/users_2021.parquet']
s3://bronze/users/2021/users_2021.parquet written (111.7 MB)
